# 04 - Evaluate LLaMEA Champions (N=10 Independent Benchmark Runs)

This notebook executes the official empirical benchmark protocol for the best synthesized algorithms:
1. Loads winning Clean and Noisy champions from `data/champions.json` (discovered across all models from `db.sqlite3`).
2. **Pre-flight Audit Dashboard**: Scans `results/evaluations/` to display exact completion status, cached runs, pending runs, and required reruns.
3. Executes each pending champion across **$N=10$ independent problem instances** using `ioh.logger.Analyzer`.
4. Outputs full convergence traces (`.json` and `.dat`) and `provenance.json` records directly into `results/evaluations/`.

In [17]:
import sys
import json
import hashlib
import shutil
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import ioh

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from core.config import DATA_DIR, PROJECT_ROOT, RESULTS_DIR
from domain.services.noise_strategy import MultiplicativeNoiseStrategy, NoNoiseStrategy
from infra.problems.bbob import BBOBProblem
from infra.storage import get_db_connection
from synthesis.execution import AlgorithmExecutor

# ── User Execution Controls & Selective Filters ──────────────────────────────
FORCE_REEVALUATE  = False   # Set True to bypass cache and re-evaluate all
FILTER_MODELS     = None    # e.g., ['7b'], ['14b'], or None for all discovered models
FILTER_PROBLEMS   = None    # e.g., [1, 8, 11, 15, 21] or None for all
FILTER_STRATEGIES = None    # e.g., ['baseline', 'guided', 'thinking', 'vectorization'] or None
FILTER_MODES      = None    # e.g., ['clean'], ['noisy'], or None for all
FILTER_DIMS       = None    # e.g., [2, 3, 5], or None for all
N_RUNS            = 10      # Number of independent benchmark runs per configuration
TIMEOUT_SECONDS   = 10.0    # Timeout per run in seconds
BUDGET            = 1000000 # Benchmark evaluation budget limit

CHAMPIONS_PATH   = DATA_DIR / 'champions.json'
EVALUATIONS_DIR  = RESULTS_DIR / 'evaluations'
EVALUATIONS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Evaluations Directory : {EVALUATIONS_DIR}')
print(f'Champions JSON Source : {CHAMPIONS_PATH}')

Evaluations Directory : /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/evaluations
Champions JSON Source : /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/champions.json


## 1. Load Champions JSON

In [18]:
if not CHAMPIONS_PATH.exists():
    raise FileNotFoundError(f'Champions file not found at {CHAMPIONS_PATH}. Please run Notebook 03 first.')

with open(CHAMPIONS_PATH, 'r', encoding='utf-8') as f:
    champions_raw = json.load(f)

# Support both nested {model: {key: info}} and flat {key: info} formats
champions_flat = {}
for k, v in champions_raw.items():
    if isinstance(v, dict) and 'code_path' in v:
        champions_flat[k] = v
    elif isinstance(v, dict):
        for sub_k, sub_v in v.items():
            champions_flat[f'{k}/{sub_k}'] = sub_v

print(f'Loaded {len(champions_flat)} champion configuration(s) across {len(champions_raw)} model category(ies):')
for model_key, model_dict in champions_raw.items():
    if isinstance(model_dict, dict) and 'code_path' not in model_dict:
        print(f'  • {model_key:<45}: {len(model_dict):3d} champions')

Loaded 227 champion configuration(s) across 2 model category(ies):
  • qwen2.5-coder-14b-instruct-q4_k_m.gguf       : 108 champions
  • qwen2.5-coder-7b-instruct-q4_k_m.gguf        : 119 champions


## 2. Pre-Flight Diagnostic Dashboard: Coverage & Workload Audit

Scans the filesystem (`results/evaluations/`) and validates checksums against `data/champions.json` to categorize every champion as:
- **`COMPLETED`**: Valid `provenance.json` with matching SHA-256 code hash and complete `.dat` traces ($N=10$ runs).
- **`PENDING`**: Not yet evaluated.
- **`NEEDS_RERUN`**: Folder exists but files are incomplete, corrupted, or code hash changed.
- **`MISSING_CODE`**: Generated `.py` file missing on disk.

In [19]:
from IPython.display import display, HTML

def compute_code_hash(code_str: str) -> str:
    return hashlib.sha256(code_str.strip().encode('utf-8')).hexdigest()

def get_model_slug(llm_name: str) -> str:
    l = str(llm_name).lower()
    if '14b' in l:
        return 'qwen_14b'
    elif '7b' in l:
        return 'qwen_7b'
    elif '70b' in l:
        return 'qwen_70b'
    else:
        m = re.search(r'([a-zA-Z0-9]+).*?(\d+b)', l)
        if m:
            return f"{m.group(1)}_{m.group(2)}"
        return l.removesuffix('.gguf').replace('-', '_').replace('.', '_').split('/')[-1]

audit_records = []

for key, info in champions_flat.items():
    p_id      = int(info['problem_id'])
    dim       = int(info['dim'])
    mode      = info.get('mode', 'all').lower()
    strat     = info.get('prompt_strategy', 'baseline').lower()
    llm_name  = info.get('llm_name', key.split('/')[0])
    noise_std = float(info.get('noise_std', 0.0))
    algo_name = info['algorithm_name']
    clean_key = key.split('/')[-1]

    # Check filters
    is_filtered = False
    if FILTER_MODELS and not any(m.lower() in llm_name.lower() for m in FILTER_MODELS): is_filtered = True
    if FILTER_PROBLEMS and p_id not in FILTER_PROBLEMS: is_filtered = True
    if FILTER_STRATEGIES and strat not in FILTER_STRATEGIES: is_filtered = True
    if FILTER_MODES and mode not in FILTER_MODES: is_filtered = True
    if FILTER_DIMS and dim not in FILTER_DIMS: is_filtered = True

    code_file = PROJECT_ROOT / info['code_path'] if not Path(info['code_path']).is_absolute() else Path(info['code_path'])
    has_code = code_file.exists()
    code_hash = compute_code_hash(code_file.read_text(encoding='utf-8')) if has_code else ''

    model_slug = get_model_slug(llm_name)
    folder_name = f"{model_slug}_{strat}"
    target_log_folder = EVALUATIONS_DIR / f"{dim}D" / f"std_{noise_std}" / f"f{p_id}" / folder_name
    prov_path = target_log_folder / "provenance.json"

    status = 'PENDING'
    runs_found = 0
    med_err = None

    if not has_code:
        status = 'MISSING_CODE'
    elif target_log_folder.exists():
        dat_files = [f for f in target_log_folder.glob('**/*.dat') if f.stat().st_size > 0]
        runs_found = len(dat_files)
        if prov_path.exists():
            try:
                prov = json.loads(prov_path.read_text(encoding='utf-8'))
                med_err = prov.get('median_clean_error')
                if prov.get('code_hash') == code_hash and runs_found > 0:
                    status = 'COMPLETED'
                else:
                    status = 'NEEDS_RERUN'
            except Exception:
                status = 'NEEDS_RERUN'
        else:
            status = 'NEEDS_RERUN'

    audit_records.append({
        'model': model_slug,
        'llm_name': llm_name,
        'key': clean_key,
        'problem_id': p_id,
        'dim': dim,
        'noise_std': noise_std,
        'strategy': strat,
        'algorithm_name': algo_name,
        'status': status,
        'runs_found': runs_found,
        'median_error': med_err,
        'is_filtered': is_filtered,
    })

df_audit = pd.DataFrame(audit_records)

# Build summary rows
summary_rows = []
for model_name, grp in df_audit.groupby('model'):
    total = len(grp)
    completed = len(grp[grp['status'] == 'COMPLETED'])
    pending = len(grp[grp['status'] == 'PENDING'])
    needs_rerun = len(grp[grp['status'] == 'NEEDS_RERUN'])
    missing_code = len(grp[grp['status'] == 'MISSING_CODE'])
    to_run = len(grp[(grp['status'].isin(['PENDING', 'NEEDS_RERUN'])) & (~grp['is_filtered'])])
    pct = (completed / total * 100) if total > 0 else 0.0
    summary_rows.append({
        'Model': model_name,
        'Total Champions': total,
        'Completed': completed,
        'Pending': pending,
        'Needs Rerun': needs_rerun,
        'Missing Code': missing_code,
        'Queue to Run': to_run,
        'Progress (%)': pct,
    })

df_summary = pd.DataFrame(summary_rows)
total_queue = int(df_summary['Queue to Run'].sum())
total_completed = int(df_summary['Completed'].sum())
total_champs = int(df_summary['Total Champions'].sum())
overall_pct = (total_completed / total_champs * 100) if total_champs > 0 else 0.0

# Render Clean Modern HTML Visual Dashboard
html_cards = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 950px; margin: 15px 0;">
    <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 16px; border-bottom: 2px solid #E2E8F0; padding-bottom: 8px;">
        <div>
            <h2 style="margin: 0; color: #0F172A; font-size: 20px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
                🚀 Benchmark Evaluation Pre-Flight Audit
            </h2>
            <p style="margin: 4px 0 0 0; color: #64748B; font-size: 13px;">Real-time status of empirical evaluation runs across all model champions</p>
        </div>
        <div style="background: #EEF2F6; padding: 6px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; color: #334155;">
            Target: N={N_RUNS} runs / condition
        </div>
    </div>

    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 20px;">
        <div style="background: linear-gradient(135deg, #1E293B 0%, #0F172A 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #94A3B8;">Total Champions</div>
            <div style="font-size: 24px; font-weight: 700; color: #F8FAFC; margin-top: 4px;">{total_champs}</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px;">Across {len(df_summary)} Models</div>
        </div>
        <div style="background: linear-gradient(135deg, #065F46 0%, #047857 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #A7F3D0;">Completed & Valid</div>
            <div style="font-size: 24px; font-weight: 700; color: #ECFDF5; margin-top: 4px;">{total_completed}</div>
            <div style="font-size: 11px; color: #D1FAE5; margin-top: 2px;">{overall_pct:.1f}% Overall Progress</div>
        </div>
        <div style="background: linear-gradient(135deg, #C2410C 0%, #9A3412 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #FED7AA;">Queue to Run</div>
            <div style="font-size: 24px; font-weight: 700; color: #FFF7ED; margin-top: 4px;">{total_queue}</div>
            <div style="font-size: 11px; color: #FFEDD5; margin-top: 2px;">{total_queue * N_RUNS} Total Runs</div>
        </div>
        <div style="background: linear-gradient(135deg, #4338CA 0%, #3730A3 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #C7D2FE;">Est. Workload</div>
            <div style="font-size: 24px; font-weight: 700; color: #EEF2FF; margin-top: 4px;">~{total_queue * 1.5:.0f}m</div>
            <div style="font-size: 11px; color: #E0E7FF; margin-top: 2px;">@ ~1.5s per run</div>
        </div>
    </div>

    <div style="background: #F8FAFC; border: 1px solid #E2E8F0; border-radius: 10px; padding: 16px; margin-bottom: 8px;">
        <div style="font-size: 13px; font-weight: 700; color: #1E293B; margin-bottom: 12px; text-transform: uppercase; letter-spacing: 0.04em;">
            Model Evaluation Progress
        </div>
"""

for _, row in df_summary.iterrows():
    m_name = row['Model']
    tot = int(row['Total Champions'])
    comp = int(row['Completed'])
    pend = int(row['Pending'])
    rerun = int(row['Needs Rerun'])
    q = int(row['Queue to Run'])
    pct = float(row['Progress (%)'])
    color = '#10B981' if pct > 75 else '#3B82F6' if pct > 25 else '#F59E0B'
    rerun_pct = (rerun / tot * 100) if tot > 0 else 0
    
    html_cards += f"""
        <div style="margin-bottom: 14px;">
            <div style="display: flex; justify-content: space-between; font-size: 13px; font-weight: 600; color: #334155; margin-bottom: 4px;">
                <span>🤖 <strong style="color: #0F172A;">{m_name}</strong> &nbsp;({comp}/{tot} Completed)</span>
                <span style="color: {color}; font-weight: 700;">{pct:.1f}%</span>
            </div>
            <div style="background: #E2E8F0; border-radius: 6px; height: 10px; overflow: hidden; display: flex;">
                <div style="background: #10B981; width: {pct}%; transition: width 0.3s;"></div>
                <div style="background: #EF4444; width: {rerun_pct}%;"></div>
            </div>
            <div style="display: flex; gap: 14px; font-size: 11px; color: #64748B; margin-top: 5px;">
                <span>✅ Completed: <strong style="color: #059669;">{comp}</strong></span>
                <span>⏳ Pending: <strong style="color: #D97706;">{pend}</strong></span>
                <span>⚠️ Needs Rerun: <strong style="color: #DC2626;">{rerun}</strong></span>
                <span>🎯 Queue: <strong style="color: #2563EB;">{q}</strong></span>
            </div>
        </div>
    """

html_cards += """
    </div>
</div>
"""

display(HTML(html_cards))


## 3. Execute Benchmark Evaluations (N=10 Independent Runs)

Executes all pending or rerun configurations. Already completed runs with valid provenance are skipped automatically unless `FORCE_REEVALUATE=True`.

In [ ]:
executor = AlgorithmExecutor(timeout_seconds=TIMEOUT_SECONDS)

def read_provenance(prov_path: Path) -> dict | None:
    if prov_path.exists():
        try:
            return json.loads(prov_path.read_text(encoding='utf-8'))
        except Exception:
            return None
    return None

def write_provenance(prov_path: Path, info: dict, dim: int, noise_std: float, code_hash: str, med_err: float):
    prov = {
        'algorithm_name':     info['algorithm_name'],
        'experiment_id':      int(info.get('experiment_id', -1)),
        'iteration_id':       int(info.get('iteration_id', -1)),
        'code_path':          info['code_path'],
        'code_hash':          code_hash,
        'dim':                dim,
        'noise_std':          noise_std,
        'problem_id':         int(info['problem_id']),
        'prompt_strategy':    info.get('prompt_strategy', 'baseline'),
        'llm_name':           info.get('llm_name', ''),
        'mode':               info.get('mode', 'all'),
        'n_runs':             N_RUNS,
        'median_clean_error': float(med_err) if not np.isinf(med_err) else None,
        'evaluated_at':       pd.Timestamp.now().isoformat(),
    }
    prov_path.write_text(json.dumps(prov, indent=2), encoding='utf-8')

evaluated_records = []
skipped_count = 0
failed_records = []
current_model_header = None

# Filter queue items
eval_queue = [
    (k, v) for k, v in champions_flat.items()
    if not (
        (FILTER_MODELS and not any(m.lower() in str(v.get('llm_name', k)).lower() for m in FILTER_MODELS))
        or (FILTER_PROBLEMS and int(v['problem_id']) not in FILTER_PROBLEMS)
        or (FILTER_STRATEGIES and str(v.get('prompt_strategy', '')).lower() not in FILTER_STRATEGIES)
        or (FILTER_MODES and str(v.get('mode', '')).lower() not in FILTER_MODES)
        or (FILTER_DIMS and int(v['dim']) not in FILTER_DIMS)
    )
]

print(f'=== Starting Benchmark Evaluation Engine ({len(eval_queue)} candidate items in scope) ===\n')
start_time = time.time()

for idx, (key, info) in enumerate(eval_queue, start=1):
    p_id       = int(info['problem_id'])
    dim        = int(info['dim'])
    mode       = info.get('mode', 'all').lower()
    strat      = info.get('prompt_strategy', 'baseline').lower()
    llm_name   = info.get('llm_name', key.split('/')[0])
    noise_std  = float(info.get('noise_std', 0.0))
    exp_id     = int(info.get('experiment_id', -1))
    algo_name  = info['algorithm_name']
    clean_key  = key.split('/')[-1]

    if llm_name != current_model_header:
        current_model_header = llm_name
        print(f'\n📦 Model Group: [{llm_name}]')
        print('=' * 80)

    code_file = PROJECT_ROOT / info['code_path'] if not Path(info['code_path']).is_absolute() else Path(info['code_path'])
    if not code_file.exists():
        print(f'  [{idx:3d}/{len(eval_queue):3d}] ❌ MISSING CODE: {clean_key} ({code_file})')
        failed_records.append((llm_name, clean_key, 'Missing Code'))
        continue

    code_content = code_file.read_text(encoding='utf-8')
    code_hash    = compute_code_hash(code_content)

    out_dir = EVALUATIONS_DIR / f'{dim}D' / f'std_{noise_std}' / f'f{p_id}'
    model_slug = get_model_slug(llm_name)
    folder_name = f'{model_slug}_{strat}'
    target_log_folder = out_dir / folder_name
    prov_path = target_log_folder / 'provenance.json'

    # Cache Validation: Check code hash & presence of valid non-empty .dat files
    if not FORCE_REEVALUATE and target_log_folder.exists():
        prov = read_provenance(prov_path)
        dat_files = [f for f in target_log_folder.glob('**/*.dat') if f.stat().st_size > 0]
        if prov and prov.get('code_hash') == code_hash and len(dat_files) > 0:
            skipped_count += 1
            continue

    print(f'  [{idx:3d}/{len(eval_queue):3d}] ⚡ Running [{model_slug}] {clean_key} ({algo_name}, Exp #{exp_id}) across {N_RUNS} runs...')
    
    # Ensure out_dir parent exists, but remove target_log_folder so IOH creates it cleanly without appending '-1'
    out_dir.mkdir(parents=True, exist_ok=True)
    if target_log_folder.exists():
        shutil.rmtree(target_log_folder, ignore_errors=True)

    noise_strat = MultiplicativeNoiseStrategy(noise_std) if noise_std > 0.0 else NoNoiseStrategy()
    run_errors = []
    t0 = time.time()

    logger = ioh.logger.Analyzer(
        root=str(out_dir),
        folder_name=folder_name,
        algorithm_name=f'LLaMEA-{model_slug}/{strat}',
        store_positions=False
    )

    for run_idx in range(1, N_RUNS + 1):
        problem = BBOBProblem(
            problem_id=p_id,
            dim=dim,
            instance_id=run_idx,
            noise_strategy=noise_strat,
        )
        problem.attach_logger(logger)

        try:
            x_opt, f_opt = executor.execute_algorithm(
                code=code_content,
                name=algo_name,
                dim=dim,
                problem=problem,
                budget=BUDGET
            )
            clean_prob = BBOBProblem(problem_id=p_id, dim=dim, instance_id=run_idx, noise_strategy=NoNoiseStrategy())
            final_err = clean_prob(x_opt) if x_opt is not None else float(f_opt)
            run_errors.append(final_err)
        except Exception as e:
            print(f'       ❌ Run {run_idx:2d}/{N_RUNS} failed: {e}')
            run_errors.append(float('inf'))
        finally:
            if hasattr(problem, 'clean_problem') and hasattr(problem.clean_problem, 'detach_logger'):
                problem.clean_problem.detach_logger()

    del logger
    med_err = np.median(run_errors) if run_errors else float('inf')
    write_provenance(prov_path, info, dim, noise_std, code_hash, med_err)
    elapsed = time.time() - t0
    
    evaluated_records.append({
        'model': model_slug,
        'key': clean_key,
        'problem_id': p_id,
        'dim': dim,
        'noise_std': noise_std,
        'strategy': strat,
        'algorithm_name': algo_name,
        'median_error': med_err,
        'runtime_sec': elapsed,
        'status': 'SUCCESS' if not np.isinf(med_err) else 'FAILED_RUNS',
    })
    print(f'       ✅ Done in {elapsed:.1f}s | Median Error: {med_err:.4e}')

total_elapsed = time.time() - start_time
print('\n' + '='*80)
print(f'🎯 Evaluation Phase Complete in {total_elapsed:.1f}s ({len(evaluated_records)} evaluated, {skipped_count} skipped/cached, {len(failed_records)} errors)')
print('👉 Run Notebook 06 (06_experimental_audit.ipynb) for the full matrix audit.')
print('='*80)


=== Starting Benchmark Evaluation Engine (227 candidate items in scope) ===


📦 Model Group: [qwen2.5-coder-14b-instruct-q4_k_m.gguf]

📦 Model Group: [qwen2.5-coder-7b-instruct-q4_k_m.gguf]
  [146/227] ⚡ Running [qwen_7b] f8_3D_noisy_thinking (NoiseResilientOptimizer, Exp #208) across 10 runs...
       ✅ Done in 28.1s | Median Error: 2.2442e+03
  [147/227] ⚡ Running [qwen_7b] f8_3D_clean_vectorization (NovelMetaheuristicImproved, Exp #193) across 10 runs...
       ✅ Done in 23.9s | Median Error: -9.8344e+00
  [148/227] ⚡ Running [qwen_7b] f8_3D_noisy_vectorization (MultiEvalEvolutionaryAlgorithm, Exp #266) across 10 runs...
       ✅ Done in 14.2s | Median Error: -9.7554e+00
  [149/227] ⚡ Running [qwen_7b] f8_5D_clean_baseline (AdaptiveGradientDescentSimulatedAnnealingHillClimber, Exp #276) across 10 runs...
       ✅ Done in 52.8s | Median Error: -6.3126e+00
  [150/227] ⚡ Running [qwen_7b] f8_5D_noisy_baseline (NoisyOptimizer, Exp #243) across 10 runs...
       ✅ Done in 14.0s | Median 